# Project FORESIGHT — Forecasting & Model Selection

Seasonal-Naive baseline, Holt comparison, feature-based ML experiment and rolling-origin backtesting.

In [1]:
from pathlib import Path
import pandas as pd, json
ROOT=Path.cwd().resolve().parent
comparison=pd.read_csv(ROOT/"reports/model_comparison.csv")
backtest=pd.read_csv(ROOT/"reports/rolling_backtest_results.csv")
summary=json.loads((ROOT/"reports/forecast_selection_summary.json").read_text())
forecast=pd.read_csv(ROOT/"data/processed/demand_forecast_results.csv")
print("Selected model:", summary["final_model"])
print("Rolling WAPE:", summary["final_model_rolling_wape"])
display(comparison)

Selected model: HistGradientBoostingRegressor
Rolling WAPE: 10.237003911218885


,method,single_holdout_wape,rolling_average_wape,role
0,Seasonal Naive (Baseline),11.623291,10.840223,Baseline / robust operational fallback
1,Holt Exponential Smoothing,10.564259,15.189860,Classical time-series comparison
2,HistGradientBoostingRegressor,8.959786,10.237004,Final selected model


In [2]:
display(backtest)
print("Relative WAPE improvement vs Seasonal Naive:", round(summary["relative_wape_improvement_percent"],2), "%")

,fold,test_start,test_end,rows,seasonal_naive_wape,holt_wape,ml_wape
0,1,2025-01-06,2025-03-31,650,10.096888,17.693501,14.730989
1,2,2025-04-07,2025-06-30,650,9.800621,13.523456,8.123785
2,3,2025-07-07,2025-09-29,650,11.840094,18.978221,9.133456
3,4,2025-09-29,2025-12-22,650,11.623291,10.564259,8.959786


Relative WAPE improvement vs Seasonal Naive: 5.56 %


In [3]:
print("Final forecast rows:", len(forecast))
print("Future period:", forecast["forecast_week"].min(), "to", forecast["forecast_week"].max())
print("Method:", forecast["forecast_method"].unique())
display(forecast.head(10))

Final forecast rows: 650
Future period: 2026-01-05 to 2026-03-30
Method: ['HistGradientBoostingRegressor']


,sku,product_name,category,subcategory,forecast_week,reference_week,forecast_demand,forecast_method,latest_inventory_snapshot_date,latest_known_on_hand_units,latest_known_on_order_units,lead_time_days,safety_stock,reorder_point,latest_known_inventory_value,cost_price,selling_price,gross_margin_per_unit
0,SKU001,Product 001,Furniture,Chair,2026-01-05,2025-01-06,100.0,HistGradientBoostingRegressor,2025-12-01,771,147,4,159,289,2815160.01,1758.45,3664.05,1905.60
1,SKU002,Product 002,Home Decor,Table,2026-01-05,2025-01-06,63.0,HistGradientBoostingRegressor,2025-12-01,526,29,3,94,149,2974319.60,3867.09,3805.69,-61.40
2,SKU003,Product 003,Kitchen,Cushion,2026-01-05,2025-01-06,34.0,HistGradientBoostingRegressor,2025-12-01,349,12,3,88,123,2145941.67,589.48,8078.23,7488.75
3,SKU004,Product 004,Lighting,Cookware,2026-01-05,2025-01-06,35.0,HistGradientBoostingRegressor,2025-12-01,259,254,6,98,196,321905.92,1445.74,6861.57,5415.83
4,SKU005,Product 005,Storage,Lamp,2026-01-05,2025-01-06,97.0,HistGradientBoostingRegressor,2025-12-01,471,291,6,129,245,3500674.53,5543.63,9493.22,3949.59
5,SKU006,Product 006,Furniture,Shelf,2026-01-05,2025-01-06,50.0,HistGradientBoostingRegressor,2025-12-01,359,485,13,129,340,2378877.60,6021.91,8106.04,2084.13
6,SKU007,Product 007,Home Decor,Cabinet,2026-01-05,2025-01-06,171.0,HistGradientBoostingRegressor,2025-12-01,207,460,8,98,237,941442.21,7748.20,5114.09,-2634.11
7,SKU008,Product 008,Kitchen,Rug,2026-01-05,2025-01-06,105.0,HistGradientBoostingRegressor,2025-12-01,325,155,3,91,161,2165845.50,2775.15,9842.64,7067.49
8,SKU009,Product 009,Lighting,Organizer,2026-01-05,2025-01-06,55.0,HistGradientBoostingRegressor,2025-12-01,75,89,7,63,131,224818.50,3121.06,2336.89,-784.17
9,SKU010,Product 010,Storage,Sofa,2026-01-05,2025-01-06,118.0,HistGradientBoostingRegressor,2025-12-01,83,70,13,22,60,595250.27,3889.06,663.46,-3225.60


## Selection decision
The feature-based HistGradientBoostingRegressor is selected because it achieved the lowest rolling-origin average WAPE among the evaluated methods. The selection is based on the same four 13-week folds used for the project benchmark, with no random train/test split.